In [4]:
import ee
ee.Initialize()

In [3]:
import json
import pandas as pd

def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer de 56 metros alrededor del punto = ~1 hectárea
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()
    
    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    
    depths = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm']  # Profundidades en centímetros 60-100cm_mean
    
    results = {}
    
    for prop, collection_id in properties.items():
        results[prop] = {}
        try:
            image = ee.Image(collection_id)
            
            for depth in depths:
                band_name = f"{prop}_{depth}_mean"
                
                # Usamos Reducer.mean() sobre la región completa para cada profundidad
                val = image.reduceRegion(
                    reducer=ee.Reducer.mean(), 
                    geometry=region,
                    scale=250, # Resolución nativa de SoilGrids
                    bestEffort=True
                ).get(band_name)
                
                results[prop][depth] = val.getInfo()
                
        except Exception as e:
            results[prop] = f"Error: {e}"
            
    return results

# Probamos con el área de 1 hectárea
# Finca Matanza 7.300921,-73.009794
soil_data = get_soil_profile_area(7.3297, -73.1867)
# print("Perfil de suelo completo por profundidades (1ha):")
# print(json.dumps(soil_data, indent=4))

# Assuming 'soil_data' is your nested dictionary result
df = pd.DataFrame(soil_data).T
df.index.name = 'property'
df.reset_index(inplace=True)
print(df.head(10))

# Save to CSV
df.to_csv("../databases/soil_profile_data_02.csv", index=False)
print("CSV export successful!")

  property       0-5cm      5-15cm     15-30cm     30-60cm    60-100cm
0    phh2o   52.509804   53.254902   53.000000   53.254902   54.254902
1      soc  568.901961  387.764706  222.705882  161.568627  168.019608
2     clay  344.470588  351.941176  375.215686  427.686275  418.431373
3     sand  312.019608  314.235294  311.470588  280.960784  276.450980
4     silt  344.254902  334.568627  313.568627  291.098039  305.117647
5     bdod   99.254902  102.509804  107.764706  112.509804  117.254902
6      cec  186.901961  184.803922  168.294118  162.784314  159.039216
CSV export successful!


In [5]:
import json
from datetime import datetime
from pathlib import Path
import pandas as pd


def get_soil_profile_area(lat, lon, area_meters=56):
    # Definimos el área: un buffer de 56 metros alrededor del punto = ~1 hectárea
    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    properties = {
        'phh2o': 'projects/soilgrids-isric/phh2o_mean',
        'soc': 'projects/soilgrids-isric/soc_mean',
        'clay': 'projects/soilgrids-isric/clay_mean',
        'sand': 'projects/soilgrids-isric/sand_mean',
        'silt': 'projects/soilgrids-isric/silt_mean',
        'bdod': 'projects/soilgrids-isric/bdod_mean',
        'cec': 'projects/soilgrids-isric/cec_mean'
    }
    depths = ['0-5cm', '5-15cm', '15-30cm', '30-60cm', '60-100cm']  # Profundidades en centímetros

    results = {}
    for prop, collection_id in properties.items():
        results[prop] = {}
        try:
            image = ee.Image(collection_id)
            for depth in depths:
                band_name = f"{prop}_{depth}_mean"
                # Usamos Reducer.mean() sobre la región completa para cada profundidad
                val = image.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=region,
                    scale=250,  # Resolución nativa de SoilGrids
                    bestEffort=True
                ).get(band_name)
                results[prop][depth] = val.getInfo()
        except Exception as e:
            results[prop] = f"Error: {e}"

    return results


def save_soil_profile(soil_data, out_prefix="soil_profile_data", output_dir="../databases"):
    """
    Guarda el DataFrame del perfil de suelo con timestamp en el nombre,
    igual que los mapas: {out_prefix}-vYYMMDDHHMMSS.csv
    """
    df = pd.DataFrame(soil_data).T
    df.index.name = 'property'
    df.reset_index(inplace=True)

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path, df


if __name__ == "__main__":
    # Finca Matanza 7.300921,-73.009794
    soil_data = get_soil_profile_area(7.300921,-73.009794)

    out_path, df = save_soil_profile(soil_data)
    print(df.head(10))

CSV guardado en ../databases/soil_profile_data-v260804201443.csv (7x6)
  property   0-5cm  5-15cm  15-30cm  30-60cm  60-100cm
0    phh2o   57.00   56.50    56.00    57.00     57.00
1      soc  633.18  389.30   279.54   184.04    153.26
2     clay  352.00  354.50   392.50   444.50    454.00
3     sand  309.00  303.50   287.50   268.50    257.50
4     silt  339.00  342.00   320.00   287.00    288.50
5     bdod  100.32  103.92   108.90   111.70    114.80
6      cec  209.00  181.50   173.50   167.50    166.50
